# Lecture notes with Qwen3-8B on a free Colab T4

**What this notebook does:** it takes a lecture transcript and turns it into short, knowledge-only study notes (Markdown), using an open model that runs on Colab's own GPU. No API key and no server.

**The pipeline (same idea as our real project):**

1. **Digest:** cut the transcript into chunks. For each chunk the model pulls out only the teachable facts (no intros, no "like and subscribe").
2. **Compose:** the model turns those facts into notebook-style sections in Markdown.
3. **Front and back matter:** a short summary, a glossary and a recap.
4. Save `notes.md` and download it.

**How to run:** first choose a GPU (**Runtime > Change runtime type > T4 GPU**). Then run the cells from top to bottom. With the built-in sample transcript the whole thing is a quick smoke test.

**Where the model comes from (two engines, chosen automatically):**

- **Engine A, "llama":** the model file `Qwen3-8B-Q4_K_M.gguf` that is already in your Google Drive (`MyDrive/AQ-Worker/models/`). Nothing big is downloaded, and it is the faster engine. It needs a prebuilt GPU wheel of `llama-cpp-python`; those only exist for some Python versions, so the install cell checks first and never starts a slow compile.
- **Engine B, "hf":** if the Drive file or the wheel is not available, we download `Qwen/Qwen3-8B` from Hugging Face (about 16 GB, once per session) and run it in 4-bit with `transformers` and `bitsandbytes`.

The cell after the GPU check prints which engine it picked and why.

## Step 1: Check the GPU, find the model in Drive, install what is needed

This cell does four things:

1. **GPU check.** If Colab gave you no GPU it stops with a clear message.
2. **Connect Google Drive.** A pop-up asks for permission; click **Allow**. Then we look for `Qwen3-8B-Q4_K_M.gguf` in `MyDrive/AQ-Worker/models/` (and, if it is not there, anywhere under `MyDrive/AQ-Worker/`). If we cannot find it we print what the folder does contain.
3. **Pick the engine.** If the file exists we try to install `llama-cpp-python` from a prebuilt GPU wheel (`--only-binary`, so pip is never allowed to compile it). If that works and the GPU is usable, the engine is **llama**.
4. **Otherwise** the engine is **hf**: we install `transformers`, `accelerate` and `bitsandbytes` (all plain Python packages).

If the last line asks you to restart the session, do it (**Runtime > Restart session**) and run this cell once more.

In [ ]:
import glob, os, subprocess, sys

# ---- 1. is there a GPU? -------------------------------------------------
try:
    import torch
except Exception as e:
    raise RuntimeError("PyTorch is missing. This notebook is meant for Google Colab.") from e

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. Use the menu: Runtime > Change runtime type > Hardware accelerator = T4 GPU, "
        "press Save, then run this cell again."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}, {vram_gb:.1f} GB memory | Python {sys.version.split()[0]} | torch {torch.__version__}")

# ---- 2. Drive: is the model file already there? --------------------------
GGUF_NAME = "Qwen3-8B-Q4_K_M.gguf"
DRIVE_MODELS = "/content/drive/MyDrive/AQ-Worker/models"
gguf_path = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    candidate = os.path.join(DRIVE_MODELS, GGUF_NAME)
    if os.path.exists(candidate):
        gguf_path = candidate
    else:
        hits = sorted(glob.glob("/content/drive/MyDrive/AQ-Worker/**/*Qwen3*8B*.gguf", recursive=True))
        gguf_path = hits[0] if hits else None
    if gguf_path:
        print(f"Found the model in Drive: {gguf_path} ({os.path.getsize(gguf_path) / 1e9:.1f} GB)")
    else:
        print("The model file was NOT found in Drive. What is in AQ-Worker:")
        for p in sorted(glob.glob("/content/drive/MyDrive/AQ-Worker/**/*", recursive=True))[:40]:
            print("  ", p)
except Exception as e:
    print(f"Drive not available ({type(e).__name__}: {str(e)[:120]}). Continuing without it.")

# ---- 3. engine A: llama.cpp from a PREBUILT wheel (never compile) ---------
ENGINE = None
if gguf_path:
    for tag in ("cu124", "cu123", "cu122", "cu121"):
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python",
             "--only-binary=:all:", "--extra-index-url", f"https://abetlen.github.io/llama-cpp-python/whl/{tag}"],
            capture_output=True, text=True,
        )
        if r.returncode == 0:
            print(f"Installed llama-cpp-python from the {tag} wheel index.")
            break
    else:
        print("No prebuilt llama-cpp-python wheel matches this Python version, so engine A is not possible.")
        print("(last pip message: " + (r.stderr.strip().splitlines() or ['?'])[-1][:200] + ")")
    # a quick, separate check that the GPU build really loads (a broken wheel must not kill this notebook)
    chk = subprocess.run(
        [sys.executable, "-c", "import llama_cpp; print(llama_cpp.llama_supports_gpu_offload())"],
        capture_output=True, text=True,
    )
    if chk.returncode == 0 and chk.stdout.strip().endswith("True"):
        ENGINE = "llama"
    else:
        print("llama-cpp-python is not usable on the GPU here:", (chk.stdout + chk.stderr).strip()[-200:])

# ---- 4. engine B: transformers + bitsandbytes -----------------------------
if ENGINE is None:
    ENGINE = "hf"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers>=4.51.0", "accelerate>=1.0.0", "bitsandbytes>=0.46.1"],
        check=True,
    )
    from importlib.metadata import version
    from packaging.version import Version
    print("transformers:", version("transformers"), "| bitsandbytes:", version("bitsandbytes"))
    if "transformers" in sys.modules and Version(sys.modules["transformers"].__version__.split("+")[0]) < Version("4.51.0"):
        print("\nRESTART NEEDED: an old transformers was already loaded. Runtime > Restart session, then run this cell again.")

print("\n>>> ENGINE:", ENGINE, "(llama = model from Drive, hf = download from Hugging Face)")

## Step 2: Load the model

**Engine llama (Drive file):** `Llama(model_path=..., n_gpu_layers=-1, n_ctx=8192)`.

- `n_gpu_layers=-1` means "put every layer on the GPU".
- `n_ctx=8192` is how much text (prompt + answer) the model can hold at once.
- Loading a 5 GB file from Drive takes a minute or two the first time.

**Engine hf (download):** we load **Qwen/Qwen3-8B** in 4-bit (`nf4`), doing the maths in `float16` because the T4 does not support `bfloat16` well. The first time Colab downloads about 16 GB (a few minutes; it vanishes when the session ends). If the 8B model does not fit, we free the memory and load **Qwen/Qwen3-4B** instead. A warning about `HF_TOKEN` can be ignored: these models are public.

In [ ]:
import gc, time

MODEL_NAME = None

if ENGINE == "llama":
    from llama_cpp import Llama
    t0 = time.time()
    llm = Llama(model_path=gguf_path, n_gpu_layers=-1, n_ctx=8192, verbose=False)
    MODEL_NAME = os.path.basename(gguf_path) + " (llama.cpp, from Drive)"
    print(f"Loaded {MODEL_NAME} in {time.time() - t0:.0f} s")
else:
    from packaging.version import Version
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    import transformers

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # transformers renamed the argument "torch_dtype" to "dtype" in version 4.56
    dtype_kwarg = "dtype" if Version(transformers.__version__.split("+")[0]) >= Version("4.56.0") else "torch_dtype"

    def load_model(name):
        t0 = time.time()
        tok = AutoTokenizer.from_pretrained(name)
        mdl = AutoModelForCausalLM.from_pretrained(
            name, quantization_config=bnb_config, device_map={"": 0}, **{dtype_kwarg: torch.float16},
        )
        mdl.eval()
        print(f"Loaded {name} in {time.time() - t0:.0f} s")
        return tok, mdl

    try:
        MODEL_NAME = "Qwen/Qwen3-8B"
        tokenizer, model = load_model(MODEL_NAME)
    except Exception as err:
        print(f"Could not load {MODEL_NAME}: {type(err).__name__}: {str(err)[:300]}")
        print("Freeing memory and trying Qwen/Qwen3-4B instead...")
        model = None
        gc.collect()
        torch.cuda.empty_cache()
        MODEL_NAME = "Qwen/Qwen3-4B"
        tokenizer, model = load_model(MODEL_NAME)

print(f"Model in use: {MODEL_NAME}")
print(f"GPU memory used by PyTorch: {torch.cuda.memory_allocated() / 1e9:.1f} GB of {vram_gb:.1f} GB (llama.cpp keeps its own memory, so this can show 0)")

## Step 3: A helper to talk to the model, and a hello test

We will call the model many times, so we write the "talking" part once, as a function called `ask`.

What `ask` does:

1. Builds the chat messages: a **system** message (the rules) and a **user** message (the task).
2. Turns off Qwen3's "think out loud" mode, because thinking is slow and wastes tokens. With engine **hf** we pass `enable_thinking=False` to the chat template. With engine **llama** we add the soft switch `/no_think` to the task text; the model then writes an empty `<think></think>` block, which we cut out afterwards.
3. Generates the answer. `temperature=0.3` means "stay steady, do not get creative".
4. Measures how long it took and how many **tokens** (word pieces) the model wrote, so we can print **tokens per second**.

`max_new_tokens` is a safety limit: the model can never run forever. If the answer hits the limit it was probably cut off, and `ask` tells you.

Every call is also written into a list called `STATS`, so at the end we can print a timing table.

In [ ]:
import re, time

STATS = []   # one entry per model call: (step name, tokens written, seconds)

def ask(system_text, user_text, max_new_tokens=1024, temperature=0.3, step="call"):
    t0 = time.time()
    if ENGINE == "llama":
        res = llm.create_chat_completion(
            messages=[{"role": "system", "content": system_text},
                      {"role": "user", "content": user_text + " /no_think"}],
            max_tokens=max_new_tokens, temperature=temperature, top_p=0.8, top_k=20,
        )
        text = res["choices"][0]["message"]["content"]
        n_in, n_out = res["usage"]["prompt_tokens"], res["usage"]["completion_tokens"]
    else:
        messages = [{"role": "system", "content": system_text}, {"role": "user", "content": user_text}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        n_in = inputs["input_ids"].shape[1]
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=True, temperature=temperature, top_p=0.8, top_k=20)
        new_tokens = out[0][n_in:]
        n_out = int(new_tokens.shape[0])
        text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    seconds = time.time() - t0

    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()   # remove the (empty) think block

    STATS.append((step, n_out, seconds))
    cut = n_out >= max_new_tokens
    print(f"  [{step}] prompt {n_in} tok, wrote {n_out} tok in {seconds:.1f} s = {n_out / max(seconds, 1e-9):.1f} tok/s"
          + ("  (hit the length limit: answer may be cut off)" if cut else ""))
    return text

hello = ask("You are a friendly assistant.", "Say hello in one short sentence.", max_new_tokens=40, step="hello")
print("\nModel says:", hello)

## Step 4: The prompts

These are copied (in shortened form) from our real project, so the notebook is self-contained.

- **`DIGEST_SYSTEM` and `digest_prompt`:** stage 1. The model reads ONE chunk and returns a small JSON object with only the teachable facts: topics, facts, definitions, steps, an example, code that was actually shown. Intros, sponsors and jokes must be dropped.
- **`NOTES_RULES` and `compose_prompt`:** stage 2. The model gets the digests and writes notebook sections in Markdown. The rules say: **knowledge only**, **faithful** (no outside facts), **rewrite, never copy**, and every section ends with key takeaways and one recall question.
- **`front_prompt`:** stage 3. From the section titles and takeaways it writes a big idea, a summary, a glossary and a recap.

The most important rule is "do not add facts that are not in the transcript". Small models like to fill gaps from memory, so always spot-check the notes against the transcript.

In [ ]:
import json

DIGEST_SYSTEM = """You are the first stage of a lecture-note pipeline. You read ONE slice of a transcript and extract ONLY its teachable knowledge as compact, faithful facts that a second stage will turn into notebook pages.

Drop intros, sponsors, subscribe asks, filler, jokes, tangents and anecdotes with no teaching value. Do not invent or add outside facts; fix only obvious auto-caption spelling of technical terms. Write facts in your own short words (no long verbatim quotes), in English. Keep numbers, names, commands, versions and code exactly as stated. Use the [mm:ss] markers for "start" when they exist.
Output ONE JSON object only, no commentary, no code fences."""

DIGEST_SHAPE = ('{"topics":[{"heading":"short topic name","start":"mm:ss or empty",'
                '"facts":["one self-contained fact per string"],'
                '"terms":[{"term":"x","definition":"y"}],'
                '"steps":["ordered step, only if a process was explained"],'
                '"example":"one illustrative example or null",'
                '"code":[{"language":"py","code":"only code actually shown"}]}]}')

def digest_prompt(chunk_text, index, total, title):
    return (
        "Lecture: " + title + "\n"
        "This is slice " + str(index + 1) + " of " + str(total) + ". (Slices overlap slightly: ignore a cut-off first or last sentence.)\n\n"
        "Extract the teachable knowledge as 1-6 topics, in order. Keep the whole answer under about 250 words: "
        "prefer facts, definitions, steps, numbers and code over narration. "
        'If the slice has NO teachable content (pure intro, outro or sponsor), return {"topics":[]}.\n\n'
        "JSON shape (leave out empty optional fields):\n" + DIGEST_SHAPE + "\n\n"
        'TRANSCRIPT SLICE:\n"""\n' + chunk_text + '\n"""'
    )

def digest_to_text(obj):
    """Turn the digest JSON into compact plain text for the compose stage."""
    topics = obj.get("topics") if isinstance(obj, dict) else None
    if not isinstance(topics, list):
        return ""
    lines = []
    for t in topics:
        if not isinstance(t, dict):
            continue
        lines.append("## [" + str(t.get("start") or "?") + "] " + str(t.get("heading") or "topic"))
        for f in t.get("facts") or []:
            lines.append("- " + str(f))
        for tm in t.get("terms") or []:
            if isinstance(tm, dict):
                lines.append("- TERM " + str(tm.get("term")) + ": " + str(tm.get("definition")))
        if t.get("steps"):
            lines.append("STEPS: " + " | ".join(str(s) for s in t["steps"]))
        if t.get("example"):
            lines.append("EXAMPLE: " + str(t["example"]))
        for c in t.get("code") or []:
            if isinstance(c, dict) and c.get("code"):
                lines.append("CODE(" + str(c.get("language") or "text") + "): " + str(c["code"])[:600])
    return "\n".join(lines)

def parse_json(text):
    """Cut out the part between the first { and the last } and parse it. Returns None if broken."""
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        return None
    try:
        return json.loads(text[a:b + 1])
    except Exception:
        return None

NOTES_RULES = """You write STUDY NOTES from lecture material, in the voice of a diligent student's field notebook: terse, concrete, well organised, with lower-case friendly section titles.

HARD RULES
1. KNOWLEDGE ONLY. Keep only what teaches something: concepts, definitions, mechanisms, steps, trade-offs, numbers, examples that illuminate, code that was actually shown. Never write about greetings, sponsors, promos, jokes or filler.
2. FAITHFUL. Every claim, number, name, command and code line must come from the material you are given. Do NOT add outside facts, do not improve the lecture with your own knowledge, do not guess what was unclear (leave it out).
3. REWRITE, never copy. Compress into clear note prose and bullets. Write facts directly ("TCP retransmits lost segments"), never "the speaker says" or "in this video".
4. Write in English. Output Markdown only (no JSON, no code fences around the whole answer).
5. STRUCTURE of every section, exactly:
## <section title> `[mm:ss]`   (use the [mm:ss] marker where the topic starts; leave the marker out if there is none)
<3-7 blocks mixing short paragraphs, bullet lists, numbered steps for processes, a Markdown table when comparing 2+ things, a **term**: definition line, a fenced code block ONLY for code that was shown>
**Key takeaways**
- <2-4 short lines>
**Recall:** Q: <a question an examiner would ask about THIS section> A: <1-3 sentence answer fully supported by the section>"""

def compose_prompt(digest_text, title, part_no, part_total, prior_titles):
    prior = ""
    if prior_titles:
        prior = ("Sections already written earlier (continue after them, do NOT repeat their topics):\n"
                 + "\n".join("- " + t for t in prior_titles[-25:]) + "\n\n")
    return (
        "Lecture: " + title + "\n"
        "This is part " + str(part_no) + " of " + str(part_total) + ". Below are condensed, already-filtered fact digests for THIS part only. Turn them into notebook sections.\n\n"
        + prior +
        "Write 1-5 sections (about one notebook page each), in order, covering the whole part from the first digest to the last. "
        "Use only the facts below. Output the sections only: no title, no introduction, no closing remarks.\n\n"
        'DIGESTS:\n"""\n' + digest_text + '\n"""'
    )

def front_prompt(title, outline_text):
    return (
        "Lecture: " + title + "\n"
        "All sections of the notes are written. Below is their outline (title and key takeaways). Using ONLY this information, write:\n"
        "1. One line starting with the characters '> ' : the big idea in at most 60 characters.\n"
        "2. A paragraph of 2-3 sentences: what a reader learns.\n"
        "3. A line containing only =====\n"
        "4. '## Glossary' followed by 6-15 bullets '- **term**: one-sentence definition' (skip a term if you are not certain of its definition).\n"
        "5. '## Recap' followed by 4-8 bullets that connect the sections.\n"
        "No other text.\n\n"
        "OUTLINE:\n" + outline_text
    )

print("Prompts ready.")

## Step 5: Your transcript, and cutting it into chunks

**Choose one mode** by editing `MODE` in the cell below:

- `"sample"`: a tiny built-in transcript about bread-making. Use this first, it is the smoke test.
- `"paste"`: paste your own transcript between the triple quotes in `PASTED_TEXT`.
- `"upload"`: a file-picker button appears and you choose a `.txt` file from your computer.

Also fill in `TITLE` (it only helps the model understand the topic).

**Chunking:** the model can only read a limited amount at once, and small models do better on small pieces. So we cut the transcript into chunks of about `CHUNK_CHARS` characters (1 token is roughly 4 characters). We cut at the end of a sentence, and each chunk repeats the last 300 characters of the previous one so that no idea is cut in half.

**Time estimate:** on a T4 expect roughly 10 to 20 tokens per second. A 1-hour lecture is about 55,000 characters, so about 10 chunks, and takes roughly 30 to 60 minutes. To test on a long transcript first, set `MAX_CHUNKS = 3`.

In [ ]:
MODE = "sample"            # "sample", "paste" or "upload"
TITLE = "Untitled lecture" # a short title helps the model
MAX_CHUNKS = None          # e.g. 3 to only process the first 3 chunks (quick test); None = all
CHUNK_CHARS = 6000         # about 1,500 tokens per chunk
OVERLAP = 300              # characters repeated between neighbouring chunks

SAMPLE_TRANSCRIPT = """[00:00] Hi everyone, welcome back to the channel, don't forget to like and subscribe, and thanks to our sponsor for supporting this video. Okay, today we talk about how bread rises.
[00:25] Bread dough contains flour, water, salt and yeast. Yeast is a tiny living fungus. When yeast is mixed with water and flour it eats the sugars that come from the flour starch and releases carbon dioxide gas and a little alcohol. This process is called fermentation. The gas gets trapped inside the dough and that is what makes the dough puff up.
[01:10] Why does the gas stay inside the dough? Because of gluten. Gluten forms when two proteins in wheat flour, glutenin and gliadin, are mixed with water and then stretched or kneaded. Gluten is like a stretchy net. Kneading lines the proteins up and makes the net stronger, so it can hold the gas bubbles like tiny balloons. Bread flour has about 12 to 14 percent protein, while cake flour has only about 7 to 9 percent, so cake flour makes a softer, crumblier result.
[02:05] By the way, my cat loves to sit on the flour bag, she is called Mimi, that is so funny. Anyway. Temperature matters a lot. Yeast works fastest at around 35 degrees Celsius. Below 5 degrees it almost stops, which is why dough can be kept in the fridge overnight for a slow rise that gives more flavor. Above 55 degrees the yeast dies, so never mix yeast with boiling water.
[03:00] Here are the steps for a basic loaf. One, mix flour, water, salt and yeast. Two, knead for about 10 minutes until the dough is smooth and springs back when poked. Three, let it rise for about one hour until it has doubled in size. Four, shape it and let it rest for 30 minutes. Five, bake at 230 degrees Celsius for 30 to 35 minutes. During baking the gas expands one last time, which is called oven spring, then the heat kills the yeast and sets the crust.
[04:10] Salt is not only for taste. It slows the yeast down a little and strengthens the gluten. Without salt the dough rises too fast and tastes flat. That is it for today, see you in the next video, bye!"""

PASTED_TEXT = """
[00:00] Paste your transcript here. Timestamps like [mm:ss] are optional but helpful.
"""

if MODE == "sample":
    transcript = SAMPLE_TRANSCRIPT
    TITLE = "How bread rises (sample)"
    CHUNK_CHARS = 1300     # small on purpose, so the sample is cut into several chunks and tests the whole pipeline
    OVERLAP = 100
elif MODE == "paste":
    transcript = PASTED_TEXT
elif MODE == "upload":
    from google.colab import files
    uploaded = files.upload()                     # shows a "Choose files" button
    if not uploaded:
        raise RuntimeError("No file was uploaded. Run the cell again and pick a .txt file.")
    filename = next(iter(uploaded))
    transcript = uploaded[filename].decode("utf-8", errors="replace")
    print("Loaded file:", filename)
else:
    raise ValueError('MODE must be "sample", "paste" or "upload"')

transcript = re.sub(r"[ \t]+", " ", transcript).strip()
if len(transcript) < 200:
    raise ValueError("The transcript is too short (under 200 characters). Check MODE and your text.")

def chunk_text(text, size, overlap):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            # move the cut back to the nearest sentence end in the last 25% of the chunk
            lo = start + int(size * 0.75)
            cut = max(text.rfind(". ", lo, end), text.rfind("? ", lo, end), text.rfind("\n", lo, end))
            if cut > start:
                end = cut + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

chunks = chunk_text(transcript, CHUNK_CHARS, OVERLAP)
if MAX_CHUNKS:
    chunks = chunks[:MAX_CHUNKS]

print(f"Transcript: {len(transcript):,} characters (about {len(transcript) // 4:,} tokens)")
print(f"Chunks: {len(chunks)}, sizes: {[len(c) for c in chunks]}")

## Step 6: Digest each chunk (stage 1)

For every chunk we ask the model for the digest JSON. Then we check it:

- If the answer is valid JSON, we turn it into compact text with `digest_to_text`.
- If the JSON is broken (small models sometimes forget a bracket), we do **not** stop. We fall back to the first part of the raw chunk, so no knowledge is lost, only compressed less.
- If the model says the chunk has no teachable content, we skip it.

Each call prints its token count and speed. At the end you see how many chunks were digested cleanly. If many fall back, the chunks are probably too big: lower `CHUNK_CHARS`.

In [ ]:
digests = []
step_times = {}
t_stage = time.time()

for i, chunk in enumerate(chunks):
    print(f"Digest {i + 1}/{len(chunks)} ({len(chunk)} chars)")
    raw = ask(DIGEST_SYSTEM, digest_prompt(chunk, i, len(chunks), TITLE),
              max_new_tokens=900, temperature=0.2, step=f"digest {i + 1}")
    obj = parse_json(raw)
    if obj is None:
        print("  -> JSON was broken, using the raw chunk instead")
        digests.append(chunk[:3000])
        continue
    text = digest_to_text(obj)
    if text.strip():
        digests.append(text)
    else:
        print("  -> no teachable content in this chunk, skipped")

step_times["digest"] = time.time() - t_stage
print(f"\nDigest stage: {step_times['digest']:.0f} s, {len(digests)} useful digests")
if digests:
    print("\n--- first digest ---")
    print(digests[0][:1200])
else:
    raise RuntimeError("No digests were produced. Check the transcript.")

## Step 7: Compose the notes (stage 2 and 3)

**Stage 2, compose:** we pack the digests into batches of at most `COMPOSE_CHARS` characters (so the prompt stays small) and ask the model to write Markdown sections for each batch. Each batch also receives the titles of sections already written, so it does not repeat them.

**Stage 3, front and back matter:** we pull out the section titles and key takeaways with a small pattern match, and ask the model for a big idea, a summary, a glossary and a recap.

Finally we glue everything together:

`# title` + big idea + summary + all sections + glossary + recap.

This is the slowest stage because the model writes the most text here.

In [ ]:
COMPOSE_CHARS = 6000   # max digest characters per compose call

# ---- pack digests into batches -------------------------------------------
batches, current = [], ""
for d in digests:
    if current and len(current) + len(d) > COMPOSE_CHARS:
        batches.append(current)
        current = ""
    current += d + "\n"
if current.strip():
    batches.append(current)
print(f"{len(digests)} digests packed into {len(batches)} compose batch(es)")

# ---- stage 2: sections ----------------------------------------------------
t_stage = time.time()
section_parts, titles = [], []
for n, batch in enumerate(batches, start=1):
    print(f"Compose {n}/{len(batches)}")
    md_text = ask(NOTES_RULES, compose_prompt(batch, TITLE, n, len(batches), titles),
                  max_new_tokens=1800, temperature=0.3, step=f"compose {n}")
    md_text = re.sub(r"^```(?:markdown|md)?\s*\n|\n```\s*$", "", md_text.strip())   # remove a wrapping code fence if present
    section_parts.append(md_text)
    titles += [re.sub(r"\s*`\[[^\]]*\]`\s*$", "", t).strip() for t in re.findall(r"^##\s+(.+)$", md_text, flags=re.M)]
sections_md = "\n\n".join(section_parts)
step_times["compose"] = time.time() - t_stage

# ---- stage 3: big idea, summary, glossary, recap -------------------------
t_stage = time.time()
outline = []
for block in re.split(r"(?m)^(?=##\s)", sections_md):
    head = re.match(r"##\s+(.+)", block)
    if not head:
        continue
    take = re.search(r"\*\*Key takeaways\*\*\s*(.*?)(?=\n\*\*Recall|\Z)", block, flags=re.S)
    outline.append("- " + head.group(1).strip() + (" | " + " ".join(take.group(1).split()) if take else ""))
front_raw = ask(NOTES_RULES.split("HARD RULES")[0], front_prompt(TITLE, "\n".join(outline) or sections_md[:3000]),
                max_new_tokens=900, temperature=0.3, step="front/back matter")
step_times["front/back matter"] = time.time() - t_stage

if "=====" in front_raw:
    top, bottom = front_raw.split("=====", 1)
else:
    top, bottom = "", front_raw
notes_md = "# " + TITLE + "\n\n" + (top.strip() + "\n\n" if top.strip() else "") + sections_md.strip() + "\n\n" + bottom.strip() + "\n"
print(f"\nNotes: {len(notes_md):,} characters, {len(titles)} sections")

## Step 8: Read it, check the speed, save and download

First we show the finished notes (rendered as formatted Markdown), then a timing table: how long each step took and how fast the model wrote (**tokens per second**: 10 to 20 is normal on a T4, below 3 means something is wrong with the GPU).

Then we save `notes.md` into Colab's disk and start a browser download. If the download pop-up is blocked, open the folder icon on the left, find `notes.md`, and download it from there.

**Check the quality yourself:** pick three facts from the notes and look for them in the transcript. Anything you cannot find was invented by the model.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(notes_md))

# ---- timing table ---------------------------------------------------------
print("\n" + "=" * 60)
print(f"{'step':<22}{'tokens':>8}{'seconds':>10}{'tok/s':>8}")
for name, tokens, secs in STATS:
    if name == "hello":
        continue
    print(f"{name:<22}{tokens:>8}{secs:>10.1f}{tokens / max(secs, 1e-9):>8.1f}")
total_tok = sum(t for n, t, s in STATS if n != "hello")
total_sec = sum(s for n, t, s in STATS if n != "hello")
print("-" * 60)
print(f"{'TOTAL generation':<22}{total_tok:>8}{total_sec:>10.1f}{total_tok / max(total_sec, 1e-9):>8.1f}")
for k, v in step_times.items():
    print(f"stage {k}: {v:.0f} s")
print(f"model: {MODEL_NAME}")

# ---- save and download ----------------------------------------------------
with open("/content/notes.md", "w", encoding="utf-8") as f:
    f.write(notes_md)
print("\nSaved /content/notes.md")

try:
    from google.colab import files
    files.download("/content/notes.md")
except Exception as e:
    print("Automatic download did not start (" + type(e).__name__ + "). Use the folder icon on the left to download notes.md.")